# Case Study 1 — Reusable vs Single-Use Bottle

Companion: **study.md**. Comparative product LCA with a break-even analysis and
Monte Carlo. Runs offline on free, hand-built data. Uses its own project
`cs1-bottle` (isolated from the tutorials).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc
import bw2io as bi

13:19:20-0400

 [

warning  

] 

Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.

## Project setup (reuse the tutorial biosphere, or install if needed)

In [2]:
PROJECT = "cs1-bottle"
if PROJECT not in bd.projects:
    # clone biosphere from the tutorial project if present, else install remotely
    if "bw25-tutorials" in bd.projects:
        bd.projects.set_current("bw25-tutorials")
        bd.projects.copy_project(PROJECT, switch=True)
    else:
        bi.remote.install_project("ecoinvent-3.10-biosphere", PROJECT)
        bd.projects.set_current(PROJECT)
else:
    bd.projects.set_current(PROJECT)

BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)
print("project:", bd.projects.current, "| biosphere:", BIOSPHERE)

def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2 = find_flow("Carbon dioxide, fossil")
so2 = find_flow("Sulfur dioxide")

project:

cs1-bottle

| biosphere:

ecoinvent-3.10-biosphere

## LCI — load the foreground from the CSV and build the database

In [3]:
csv_path = Path.cwd() / "data" / "bottle_inventory.csv"
inv = pd.read_csv(csv_path)
print(inv.to_string(index=False))

       process   exchange   input_db  amount          unit         type                                   notes
single_use_pet production foreground  1.0000          unit   production                          one PET bottle
single_use_pet       elec foreground  0.2000 kilowatt hour technosphere                          bottle forming
single_use_pet co2_fossil  biosphere  0.0820      kilogram    biosphere        PET resin + forming (per bottle)
reusable_steel production foreground  1.0000          unit   production              one steel bottle (durable)
reusable_steel       elec foreground  4.5000 kilowatt hour technosphere               steel forming + finishing
reusable_steel co2_fossil  biosphere  3.6000      kilogram    biosphere stainless steel production (per bottle)
          wash production foreground  1.0000          wash   production                       one washing cycle
          wash       elec foreground  0.0150 kilowatt hour technosphere              hot water heating p

In [4]:
DB = "cs1_fg"
if DB in bd.databases:
    del bd.databases[DB]

flow_map = {"co2_fossil": co2.key, "so2": so2.key}

data = {}
for proc in inv["process"].unique():
    rows = inv[inv["process"] == proc]
    unit = rows.iloc[0]["unit"] if (rows["type"] == "production").any() else "unit"
    prod_unit = rows[rows["type"] == "production"]["unit"].iloc[0]
    exchanges = []
    for _, r in rows.iterrows():
        if r["type"] == "production":
            exchanges.append({"input": (DB, proc), "amount": r["amount"], "type": "production"})
        elif r["type"] == "biosphere":
            exchanges.append({"input": flow_map[r["exchange"]], "amount": r["amount"], "type": "biosphere"})
        else:  # technosphere -> exchange names another foreground process
            exchanges.append({"input": (DB, r["exchange"]), "amount": r["amount"], "type": "technosphere"})
    data[(DB, proc)] = {"name": proc, "unit": prod_unit, "exchanges": exchanges}

bd.Database(DB).write(data)
print("built processes:", [a["name"] for a in bd.Database(DB)])

13:19:21-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

info     

] 

Vacuuming database            

built processes:

['reusable_steel', 'single_use_pet', 'elec', 'wash']

## LCIA — per-bottle / per-wash impacts

In [5]:
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))

def score(code, amount=1.0):
    act = bd.get_node(database=DB, code=code)
    l = bc.LCA({act: amount}, method=gwp); l.lci(); l.lcia()
    return l.score

pet_per_bottle = score("single_use_pet")
steel_mfg = score("reusable_steel")
wash_per_use = score("wash")
print(f"single-use PET  : {pet_per_bottle:.4f} kg CO2-eq / use")
print(f"steel mfg (fixed): {steel_mfg:.4f} kg CO2-eq (amortized over lifetime)")
print(f"wash per use    : {wash_per_use:.5f} kg CO2-eq / use")

single-use PET  : 0.1660 kg CO2-eq / use

steel mfg (fixed): 5.4900 kg CO2-eq (amortized over lifetime)

wash per use    : 0.00630 kg CO2-eq / use

## Interpretation — break-even analysis

In [6]:
N = np.arange(1, 501)
reusable_per_use = steel_mfg / N + wash_per_use
single_per_use = np.full_like(N, pet_per_bottle, dtype=float)

crossings = np.where(reusable_per_use <= single_per_use)[0]
break_even = int(N[crossings[0]]) if len(crossings) else None
print("break-even at N =", break_even, "uses")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(N, single_per_use, "--", color="#C44E52", label="single-use PET (per use)")
ax.plot(N, reusable_per_use, color="#55A868", label="reusable steel (amortized)")
if break_even:
    ax.axvline(break_even, color="k", lw=1, ls=":")
    ax.annotate(f"break-even ≈ {break_even} uses",
                xy=(break_even, single_per_use[0]),
                xytext=(break_even + 40, single_per_use[0] * 1.6),
                arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("number of uses of the reusable bottle")
ax.set_ylabel("GWP per use (kg CO2-eq)")
ax.set_title("Break-even: reusable vs single-use bottle")
ax.set_ylim(0, pet_per_bottle * 2.5); ax.legend()
plt.tight_layout()
plt.savefig("cs1_breakeven.png", dpi=130, bbox_inches="tight")
print("saved cs1_breakeven.png")
plt.show()

break-even at N =

35

uses

saved cs1_breakeven.png

C:\Users\derne\AppData\Local\Temp\ipykernel_61632\1611776124.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Scenario: grid decarbonization moves the break-even

In [7]:
def break_even_for_grid(grid_ci):
    # rebuild elec with a new carbon intensity
    e = bd.get_node(database=DB, code="elec")
    for exc in e.biosphere():
        if exc.input == co2:
            exc["amount"] = grid_ci; exc.save()
    bd.Database(DB).process()
    smfg = score("reusable_steel"); wpu = score("wash"); pet = score("single_use_pet")
    r = smfg / N + wpu
    cr = np.where(r <= pet)[0]
    return int(N[cr[0]]) if len(cr) else None

rows = []
for g in [0.42, 0.30, 0.20, 0.10]:
    rows.append({"grid_ci (kg CO2/kWh)": g, "break-even (uses)": break_even_for_grid(g)})
# restore baseline
break_even_for_grid(0.42)
print(pd.DataFrame(rows).to_string(index=False))

 grid_ci (kg CO2/kWh)  break-even (uses)
                 0.42                 35
                 0.30                 36
                 0.20                 38
                 0.10                 41

## Monte Carlo on the break-even (uncertain manufacturing & wash energy)

In [8]:
import math
for act in bd.Database(DB):
    for exc in act.exchanges():
        if exc["type"] == "production":
            continue
        exc["uncertainty type"] = 2
        exc["loc"] = math.log(exc["amount"]) if exc["amount"] > 0 else 0.0
        exc["scale"] = 0.20
        exc.save()
bd.Database(DB).process()

def mc_scores(code, n=300):
    act = bd.get_node(database=DB, code=code)
    l = bc.LCA({act: 1}, method=gwp, use_distributions=True); l.lci(); l.lcia()
    return np.array([l.score for _ in zip(range(n), l)])

pet_mc = mc_scores("single_use_pet")
steel_mc = mc_scores("reusable_steel")
wash_mc = mc_scores("wash")
# break-even distribution: sample-wise
be = []
for pet, smfg, wpu in zip(pet_mc, steel_mc, wash_mc):
    r = smfg / N + wpu
    cr = np.where(r <= pet)[0]
    be.append(N[cr[0]] if len(cr) else N[-1])
be = np.array(be)
print(f"break-even: median={np.median(be):.0f}, "
      f"90% CI=[{np.percentile(be,5):.0f}, {np.percentile(be,95):.0f}] uses")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(be, bins=30, color="#4C72B0")
ax.axvline(np.median(be), color="k", lw=1)
ax.set_xlabel("break-even number of uses"); ax.set_ylabel("MC frequency")
ax.set_title("Uncertainty in the break-even point (300 MC draws)")
plt.tight_layout()
plt.savefig("cs1_breakeven_mc.png", dpi=130, bbox_inches="tight")
print("saved cs1_breakeven_mc.png")
plt.show()

break-even: median=35, 90% CI=[23, 55] uses

saved cs1_breakeven_mc.png

C:\Users\derne\AppData\Local\Temp\ipykernel_61632\2779489640.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Result export + conclusion

In [9]:
summary = pd.DataFrame({
    "metric": ["single-use GWP/use", "steel mfg GWP", "wash GWP/use",
               "break-even (deterministic)", "break-even median (MC)"],
    "value": [round(pet_per_bottle, 4), round(steel_mfg, 3), round(wash_per_use, 5),
              break_even, int(np.median(be))],
})
summary.to_csv("cs1_summary.csv", index=False)
print(summary.to_string(index=False))
print("\nConclusion: the reusable bottle pays back its manufacturing burden after")
print(f"~{break_even} uses at today's grid; a cleaner grid lowers this. Reported as a")
print("range, not a point, given manufacturing uncertainty.")

                    metric   value
        single-use GWP/use  0.1660
             steel mfg GWP  5.4900
              wash GWP/use  0.0063
break-even (deterministic) 35.0000
    break-even median (MC) 35.0000


Conclusion: the reusable bottle pays back its manufacturing burden after

~35 uses at today's grid; a cleaner grid lowers this. Reported as a

range, not a point, given manufacturing uncertainty.

## Optional: ecoinvent background (guarded)

In [10]:
import os
if os.environ.get("ECOINVENT_USERNAME"):
    print("With ecoinvent installed you would replace the hand-built 'elec',")
    print("PET and steel surrogates with market processes, e.g.:")
    print("  ei = bd.Database('ecoinvent-3.10-cutoff')")
    print("  steel = ei.search('steel, chromium, 18/8')[0]")
else:
    print("No ecoinvent — study is complete on free surrogates (this is expected).")

No ecoinvent — study is complete on free surrogates (this is expected).